# Movie Recommender System â€” sklearn only
**Dataset**: MovieLens Latest Small (ml-latest-small)  
**Models**: SVD Â· NMF Â· User-based KNN  
**Library**: scikit-learn (no scikit-surprise)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neighbors import NearestNeighbors

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('All libraries imported successfully.')

## 1. Load Data

In [ ]:
DATA_DIR = 'dataset/'

ratings = pd.read_csv(DATA_DIR + 'ratings.csv')
movies  = pd.read_csv(DATA_DIR + 'movies.csv')
tags    = pd.read_csv(DATA_DIR + 'tags.csv')

print(f'Ratings : {ratings.shape[0]:,} rows')
print(f'Movies  : {movies.shape[0]:,} rows')
print(f'Tags    : {tags.shape[0]:,} rows')
ratings.head()

## 2. Exploratory Data Analysis

In [ ]:
n_users  = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)
sparsity = 1 - n_ratings / (n_users * n_movies)

print(f'Users   : {n_users}')
print(f'Movies  : {n_movies:,}')
print(f'Ratings : {n_ratings:,}')
print(f'Sparsity: {sparsity:.2%}')
print(f'Rating range: {ratings["rating"].min()} â€“ {ratings["rating"].max()}')
print(f'Mean rating : {ratings["rating"].mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Rating distribution
ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Ratings per user
user_counts = ratings.groupby('userId')['rating'].count()
axes[1].hist(user_counts, bins=40, color='coral')
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Users')

# Ratings per movie
movie_counts = ratings.groupby('movieId')['rating'].count()
axes[2].hist(movie_counts, bins=40, color='mediumseagreen')
axes[2].set_title('Ratings per Movie')
axes[2].set_xlabel('Number of Ratings')
axes[2].set_ylabel('Movies')

plt.tight_layout()
plt.show()

## 3. Train / Test Split

In [ ]:
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df):,} ratings')
print(f'Test : {len(test_df):,} ratings')

## 4. Build User-Item Matrix
Rows = users Â· Columns = movies Â· Values = ratings (0 where no rating)

In [ ]:
# Pivot on training data only
train_matrix = train_df.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Keep indices for later lookups
user_index  = {uid: i for i, uid in enumerate(train_matrix.index)}
movie_index = {mid: i for i, mid in enumerate(train_matrix.columns)}

# Global and per-user means (computed on observed ratings only)
global_mean = train_df['rating'].mean()
user_means  = train_df.groupby('userId')['rating'].mean()

print(f'Matrix shape : {train_matrix.shape}')
print(f'Global mean  : {global_mean:.4f}')

## 5. Helper â€” Evaluate on Test Set

In [ ]:
def evaluate(pred_fn, test_df, label='Model'):
    """Apply pred_fn(userId, movieId) -> float on every test row."""
    y_true, y_pred = [], []
    for _, row in test_df.iterrows():
        y_true.append(row['rating'])
        y_pred.append(pred_fn(row['userId'], row['movieId']))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f'{label:20s}  RMSE={rmse:.4f}  MAE={mae:.4f}')
    return rmse, mae, np.array(y_true), np.array(y_pred)


def precision_recall_at_k(pred_fn, test_df, k=10, threshold=3.5):
    """Precision@K and Recall@K averaged over all users."""
    user_items = defaultdict(list)
    for _, row in test_df.iterrows():
        est = pred_fn(row['userId'], row['movieId'])
        user_items[row['userId']].append((est, row['rating']))

    precisions, recalls = [], []
    for uid, pairs in user_items.items():
        pairs.sort(key=lambda x: x[0], reverse=True)
        top_k   = pairs[:k]
        n_rel   = sum(1 for _, r in pairs if r >= threshold)
        n_rec_k = sum(1 for e, _ in top_k if e >= threshold)
        n_rel_k = sum(1 for e, r in top_k if e >= threshold and r >= threshold)
        precisions.append(n_rel_k / n_rec_k if n_rec_k > 0 else 0)
        recalls.append(n_rel_k / n_rel if n_rel > 0 else 0)

    return np.mean(precisions), np.mean(recalls)

## 6. Model 1 â€” SVD (TruncatedSVD)
Mean-centre the matrix â†’ decompose â†’ reconstruct full rating predictions.

In [ ]:
# Mean-centre by user (replace 0 with NaN to compute real means, then put 0 back)
matrix_vals = train_matrix.values.copy().astype(float)
row_means   = np.true_divide(matrix_vals.sum(1), (matrix_vals != 0).sum(1))
row_means   = np.nan_to_num(row_means, nan=global_mean)

matrix_centred = matrix_vals.copy()
for i in range(matrix_centred.shape[0]):
    mask = matrix_centred[i] != 0
    matrix_centred[i, mask] -= row_means[i]

# Decompose
svd = TruncatedSVD(n_components=100, random_state=42)
U_sigma = svd.fit_transform(matrix_centred)   # shape: (n_users, 100)
Vt      = svd.components_                     # shape: (100, n_movies)

# Reconstruct and add user means back
svd_reconstructed = (U_sigma @ Vt) + row_means.reshape(-1, 1)
svd_pred_df = pd.DataFrame(
    svd_reconstructed,
    index=train_matrix.index,
    columns=train_matrix.columns
)

print('SVD trained. Explained variance ratio:', svd.explained_variance_ratio_.sum())

In [ ]:
def predict_svd(user_id, movie_id):
    if user_id in svd_pred_df.index and movie_id in svd_pred_df.columns:
        return float(np.clip(svd_pred_df.loc[user_id, movie_id], 0.5, 5.0))
    if user_id in user_means.index:
        return float(user_means[user_id])
    return global_mean

svd_rmse, svd_mae, _, _ = evaluate(predict_svd, test_df, 'SVD')

## 7. Model 2 â€” NMF (Non-negative Matrix Factorization)

In [ ]:
nmf = NMF(n_components=100, max_iter=500, random_state=42)
W   = nmf.fit_transform(train_matrix.values)   # user factors  (n_users, 100)
H   = nmf.components_                          # item factors  (100, n_movies)

nmf_reconstructed = W @ H
nmf_pred_df = pd.DataFrame(
    nmf_reconstructed,
    index=train_matrix.index,
    columns=train_matrix.columns
)

print(f'NMF trained. Reconstruction error: {nmf.reconstruction_err_:.4f}')

In [ ]:
def predict_nmf(user_id, movie_id):
    if user_id in nmf_pred_df.index and movie_id in nmf_pred_df.columns:
        return float(np.clip(nmf_pred_df.loc[user_id, movie_id], 0.5, 5.0))
    if user_id in user_means.index:
        return float(user_means[user_id])
    return global_mean

nmf_rmse, nmf_mae, _, _ = evaluate(predict_nmf, test_df, 'NMF')

## 8. Model 3 â€” User-based KNN (cosine similarity + mean-centering)

In [ ]:
K_NEIGHBORS = 40

knn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric='cosine', algorithm='brute')
knn.fit(train_matrix.values)

# Cache distances so we don't recompute per prediction
distances_all, indices_all = knn.kneighbors(train_matrix.values)
# distances_all shape: (n_users, K+1)  â€” first column is self (distance=0)

print('KNN fitted.')

In [ ]:
def predict_knn(user_id, movie_id):
    if user_id not in user_index:
        return user_means.get(user_id, global_mean)
    if movie_id not in movie_index:
        return user_means.get(user_id, global_mean)

    u_idx = user_index[user_id]
    m_idx = movie_index[movie_id]

    neighbor_idxs = indices_all[u_idx][1:]          # exclude self
    neighbor_dists = distances_all[u_idx][1:]
    neighbor_sims  = 1.0 - neighbor_dists            # cosine similarity

    neighbor_ratings = train_matrix.values[neighbor_idxs, m_idx]
    rated_mask = neighbor_ratings > 0

    if rated_mask.sum() == 0:
        return row_means[u_idx]

    sims    = neighbor_sims[rated_mask]
    ratings = neighbor_ratings[rated_mask]
    n_means = row_means[neighbor_idxs[rated_mask]]

    numerator   = np.sum(sims * (ratings - n_means))
    denominator = np.sum(np.abs(sims))

    if denominator == 0:
        return row_means[u_idx]

    pred = row_means[u_idx] + numerator / denominator
    return float(np.clip(pred, 0.5, 5.0))

knn_rmse, knn_mae, _, _ = evaluate(predict_knn, test_df, 'KNN (user-based)')

## 9. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['SVD', 'NMF', 'KNN (user-based)'],
    'RMSE' : [svd_rmse, nmf_rmse, knn_rmse],
    'MAE'  : [svd_mae,  nmf_mae,  knn_mae],
})
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['steelblue', 'coral', 'mediumseagreen']

axes[0].bar(results['Model'], results['RMSE'], color=colors)
axes[0].set_title('RMSE (lower is better)')
axes[0].set_ylabel('RMSE')

axes[1].bar(results['Model'], results['MAE'], color=colors)
axes[1].set_title('MAE (lower is better)')
axes[1].set_ylabel('MAE')

plt.tight_layout()
plt.show()

## 10. Precision@K and Recall@K

In [ ]:
# Compute for k = 5, 10, 15, 20
rows = []
for k in [5, 10, 15, 20]:
    for label, fn in [('SVD', predict_svd), ('NMF', predict_nmf), ('KNN', predict_knn)]:
        p, r = precision_recall_at_k(fn, test_df, k=k, threshold=3.5)
        rows.append({'Model': label, 'K': k, 'Precision@K': p, 'Recall@K': r})

pr_df = pd.DataFrame(rows)
print(pr_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model, grp in pr_df.groupby('Model'):
    axes[0].plot(grp['K'], grp['Precision@K'], marker='o', label=model)
    axes[1].plot(grp['K'], grp['Recall@K'],    marker='o', label=model)

axes[0].set_title('Precision@K'); axes[0].set_xlabel('K'); axes[0].legend()
axes[1].set_title('Recall@K');    axes[1].set_xlabel('K'); axes[1].legend()
plt.tight_layout()
plt.show()

## 11. Top-N Recommendations

In [ ]:
def get_top_n(user_id, pred_fn, n=10, already_rated=True):
    """
    Return top-N movie recommendations for a user.
    If already_rated=False, exclude movies the user already rated in training.
    """
    all_movies = train_matrix.columns.tolist()

    if not already_rated:
        rated_by_user = set(train_df[train_df['userId'] == user_id]['movieId'])
        all_movies = [m for m in all_movies if m not in rated_by_user]

    scores = [(mid, pred_fn(user_id, mid)) for mid in all_movies]
    scores.sort(key=lambda x: x[1], reverse=True)
    top = scores[:n]

    recs = pd.DataFrame(top, columns=['movieId', 'predicted_rating'])
    recs = recs.merge(movies[['movieId', 'title', 'genres']], on='movieId', how='left')
    return recs


# Example: top-10 for user 1 using SVD (exclude already-rated movies)
sample_user = ratings['userId'].iloc[0]
print(f'Top-10 SVD recommendations for user {sample_user}:')
get_top_n(sample_user, predict_svd, n=10, already_rated=False)

In [ ]:
print(f'Top-10 NMF recommendations for user {sample_user}:')
get_top_n(sample_user, predict_nmf, n=10, already_rated=False)

In [ ]:
print(f'Top-10 KNN recommendations for user {sample_user}:')
get_top_n(sample_user, predict_knn, n=10, already_rated=False)

## 12. Single Rating Prediction

In [ ]:
def predict_all(user_id, movie_id):
    title = movies[movies['movieId'] == movie_id]['title'].values
    title = title[0] if len(title) else f'Movie {movie_id}'
    print(f'User {user_id} x "{title}"')
    print(f'  SVD : {predict_svd(user_id, movie_id):.2f}')
    print(f'  NMF : {predict_nmf(user_id, movie_id):.2f}')
    print(f'  KNN : {predict_knn(user_id, movie_id):.2f}')

# Pick a random test row
row = test_df.iloc[0]
predict_all(int(row['userId']), int(row['movieId']))
print(f'  True: {row["rating"]:.2f}')

## 13. Model 4 — XGBoost
Frame rating prediction as regression: engineer per-(user, movie) features, train XGBoost, then generate top-N recommendations with the same interface as the other models.

In [ ]:
from xgboost import XGBRegressor

# --- Feature engineering ---
# Per-user stats from train
user_stats = train_df.groupby('userId')['rating'].agg(
    user_mean='mean', user_count='count'
).reset_index()

# Per-movie stats from train
movie_stats = train_df.groupby('movieId')['rating'].agg(
    movie_mean='mean', movie_count='count'
).reset_index()

# SVD user latent vectors (already computed) — rich signal
# U_sigma shape: (n_train_users, 100), index = train_matrix.index
svd_user_df = pd.DataFrame(
    U_sigma,
    index=train_matrix.index,
    columns=[f'svd_{i}' for i in range(U_sigma.shape[1])]
).reset_index()   # adds 'userId' column

def build_features(df):
    """Add user/movie stats and SVD latent features to a ratings DataFrame."""
    d = df.merge(user_stats,  on='userId',  how='left')
    d = d.merge(movie_stats,  on='movieId', how='left')
    d = d.merge(svd_user_df,  on='userId',  how='left')
    # Fill unknowns (cold-start)
    d['user_mean']   = d['user_mean'].fillna(global_mean)
    d['user_count']  = d['user_count'].fillna(0)
    d['movie_mean']  = d['movie_mean'].fillna(global_mean)
    d['movie_count'] = d['movie_count'].fillna(0)
    svd_cols = [f'svd_{i}' for i in range(U_sigma.shape[1])]
    d[svd_cols] = d[svd_cols].fillna(0)
    return d

feature_cols = ['user_mean', 'user_count', 'movie_mean', 'movie_count'] + \n               [f'svd_{i}' for i in range(U_sigma.shape[1])]

train_feat = build_features(train_df)
X_train = train_feat[feature_cols].values
y_train = train_feat['rating'].values

print(f'Training features shape: {X_train.shape}')

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(X_train, y_train)
print('XGBoost trained.')

In [ ]:
def predict_xgb(user_id, movie_id):
    row = pd.DataFrame([{'userId': user_id, 'movieId': movie_id}])
    feat = build_features(row)[feature_cols].values
    pred = xgb.predict(feat)[0]
    return float(np.clip(pred, 0.5, 5.0))

xgb_rmse, xgb_mae, _, _ = evaluate(predict_xgb, test_df, 'XGBoost')

### Updated Model Comparison

In [ ]:
results_full = pd.DataFrame({
    'Model': ['SVD', 'NMF', 'KNN (user-based)', 'XGBoost'],
    'RMSE' : [svd_rmse, nmf_rmse, knn_rmse, xgb_rmse],
    'MAE'  : [svd_mae,  nmf_mae,  knn_mae,  xgb_mae],
})
print(results_full.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

axes[0].bar(results_full['Model'], results_full['RMSE'], color=colors)
axes[0].set_title('RMSE (lower is better)')
axes[0].set_ylabel('RMSE')

axes[1].bar(results_full['Model'], results_full['MAE'], color=colors)
axes[1].set_title('MAE (lower is better)')
axes[1].set_ylabel('MAE')

plt.tight_layout()
plt.show()

### Top-N Recommendations with XGBoost
> **Note**: XGBoost predicts one row at a time so generating recs for all movies is slower than matrix models. For a production scenario you'd batch-score or pre-compute.

In [ ]:
print(f'Top-10 XGBoost recommendations for user {sample_user}:')
get_top_n(sample_user, predict_xgb, n=10, already_rated=False)

### Feature Importance

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
top20.plot(kind='barh', ax=ax, color='mediumpurple')
ax.set_title('XGBoost — Top 20 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()